# EL-PASO: An Open-Source Python Framework for Processing and Standardizing Particle Measurements Taken in Space

Bernhard Haas<sup>1,2</sup>, Alexander Y. Drozdov<sup>3</sup>, Mátyás Szabó-Roberts<sup>1</sup>, Sahil Jhawar<sup>1</sup>

<sup>1</sup> GFZ German Research Centre for Geosciences, Helmholtz Centre Potsdam, Potsdam, Germany  
<sup>2</sup> Institute of Physics and Astronomy, University of Potsdam, Potsdam, Germany  
<sup>3</sup> The Aerospace Corporation, El Segundo, CA, USA

**Corresponding author**: Bernhard Haas (bhaas@gfz.de)

---

*This is an executable version of the paper submitted to AGU's Earth and Space Science journal. All code can be run interactively to reproduce the results presented in the paper.*

## Key Points

- We present EL-PASO, an open-source Python framework for processing and standardizing in-situ particle measurements
- EL-PASO converts non-standardized particle data into a common data format with standardized metadata and derived products
- EL-PASO has been verified by reproducing published $L*$ values from the Van Allen Probes team and previous results obtained at GFZ

## Abstract

In this article, we present EL-PASO (ELaborative Particle Analysis from Satellite Observations), an open-source Python library for processing and standardizing in-situ particle measurements taken in space.

EL-PASO aims to address the challenges posed by the diverse data formats and metadata standards used by different space missions.
Its main purpose is to convert non-standardized particle data into a common data format with standardized metadata and to calculate derived products, such as adiabatic invariants and phase-space density.
EL-PASO supports multiple file formats, including CDF, NetCDF4, ASCII-based formats, and JSON, for both reading and saving data.

Here, we describe the software architecture and design of EL-PASO, display small code snippets as examples, and show how EL-PASO is verified against published data from other sources, such as the Van Allen Probes team and previous results obtained at GFZ.
By providing a unified library for data processing, EL-PASO facilitates the comparison and integration of particle measurements from various sources, ultimately enabling multi-mission studies at larger scales and enhancing our understanding of space weather phenomena.

## Table of Contents

1. [Introduction](#1.-Introduction)
2. [Software Architecture and Design](#2.-Software-Architecture-and-Design)
   - 2.1 [Core Data Structure](#2.1-Core-Data-Structure)
   - 2.2 [Data Extraction](#2.2-Data-Extraction)
   - 2.3 [Data Processing](#2.3-Data-Processing)
   - 2.4 [Output Standardization](#2.4-Output-Standardization)
   - 2.5 [Loading of Processed Data](#2.5-Loading-of-Processed-Data)
3. [Usability, Reproducibility, and Maintenance](#3.-Usability-Reproducibility-and-Maintenance)
   - 3.1 [Getting Started](#3.1-Getting-Started)
   - 3.2 [Reproducibility](#3.2-Reproducibility)
   - 3.3 [Metadata and FAIR Principles](#3.3-Metadata-and-FAIR-Principles)
4. [Verification and Example Applications](#4.-Verification-and-Example-Applications)
   - 4.1 [Reproducing $L*$ values obtained by the ECT team](#4.1-Reproducing-L-values-obtained-by-the-ECT-team)
   - 4.2 [Reproducing previous results obtained with other software developed at GFZ](#4.2-Reproducing-previous-results-obtained-with-other-software-developed-at-GFZ)
5. [Summary](#5.-Summary)
6. [Open Research Section](#Open-Research-Section)
7. [Acknowledgments](#Acknowledgments)
8. [References](#References)

---
# 1. Introduction

In recent decades, numerous space missions have been launched to study the near-Earth space environment. Many of these missions are equipped with instruments to measure energetic particles (electrons, protons, and ions) in situ. These measurements are crucial for understanding various space weather phenomena, including geomagnetic storms, radiation belt dynamics, and solar energetic particle events.

Missions often use different data formats, include varying metadata, and calculate different derived products, making it difficult to compare and combine datasets from different sources. To help alleviate this issue, the heliophysics community has initiated several efforts over the years, such as the development of the Space Physics Archive Search and Extract (SPASE) information model, the COSPAR Panel for Radiation Belt Environment Modeling (PRBEM) guidelines on naming conventions for radiation belt data, or the establishment of the International Heliophysics Data Environment Alliance (IHDEA). However, many mission operators do not strictly adhere to these standards, and the extensive archives of older satellite data have been retroactively made compliant only rarely, leaving researchers to deal with unharmonized datasets.
This typically requires creating substantial custom code to read, harmonize, and process data from different missions before the multi-mission analysis can even begin.

The heliophysics community developed open-source packages to address some of these challenges. The Space Physics Environment Data Analysis System (SPEDAS) (Angelopoulos et al., 2019) for the IDL programming language and the derived Python version (PySPEDAS) (Grimes et al., 2022) allow the user to download, read, and analyze space and ground-based observations with a few lines of code. Spacepy (Morley et al., 2011; Niehof et al., 2022) is a general toolbox for space physics data processing implemented in Python. Its capabilities include loading multiple file formats into a common internal data structure and using processing functions for particle analysis and radiation belt modeling. While these package address downloading and reading data from different missions, they typically do not provide output in a consistent, harmonized format. Therefore, researchers still need to write mission-specific code before the data can be used for further analysis, such as model-data comparisons. In addition, frameworks like the Heliophysics Application Programmer's Interface (HAPI) (Weigel et al., 2021) and das2 (Piker et al., 2017) provide interfaces for retrieving standardized data, but primarily in time-series format. This can be insufficient for radiation-environment analysis that involves multidimensional considerations of satellite data (e.g., data assimilation).

We identified a need within the community for a tool capable of producing standardized space physics data, including metadata, from diverse sources.
Particular priorities for the tool are the ability to operate within existing frameworks and standards, and to create datasets that fulfill the Findable-Accessible-Interoperable-Reusable (FAIR) principles (Wilkinson et al., 2016). The available processing scripts in EL-PASO significantly reduce the effort required for researchers to develop their own processing code, thereby reducing the risk of human error when rewriting code from scratch.

Therefore, we present EL-PASO (ELaborative Particle Analysis from Satellite Observations), an open-source Python library for standardizing particle measurements in space. The main goal of EL-PASO is to convert the non-standardized particle data from different missions into a common data format, with standardized metadata and derived products. The library, in its current state, is ready to process particle data from Earth's magnetospheric populations. It can be extended to other physical domains and particle populations in the future. The library is designed to be modular and extensible, allowing users to easily add support for new missions and data formats. Once a mission is processed using EL-PASO, the resulting standardized data can be comfortably read using a unified reader and combined with data from other missions, facilitating multi-mission studies and improving our understanding of space weather phenomena.

We welcome contributions to the code on GitHub at [https://github.com/GFZ/EL_PASO](https://github.com/GFZ/EL_PASO). Installation guidelines are provided on the same page. Automated documentation hosted at [el-paso.readthedocs.io/en/latest/](https://el-paso.readthedocs.io/en/latest/) is generated from the docstrings available for all public functions within EL-PASO using mkdocs (Christie et al., 2014). The package is furthermore fully type-checked using Pyright (Microsoft Corporation, 2019)  and follows the PEP8 style guides, which are enforced by Ruff (Marsh & Astral Shaper, Inc., 2022). Testing is performed by running unit- and system-level tests with Pytest (Krekel et al., 2004), which helps avoid regressions when adding new functionality to the codebase. The code is openly available under the Apache-2.0 license.

The following sections will introduce the features and internal structure of EL-PASO and demonstrate software verification by comparing results with processed data from other sources. 

---
# 2. Software Architecture and Design

EL-PASO is written in Python (compatible with versions 3.11+), utilizing other open-source libraries such as NumPy (Harris et al., 2020), SciPy (Virtanen et al., 2020), and Pandas (pandas development team, 2020), for data manipulation and analysis. The framework is structured into several modules, each responsible for a specific task in the data processing pipeline. The overall program flow is shown in Figure 1, which depicts a top-to-bottom pipeline in which raw files are iteratively transformed. Rectangular execution nodes handle sequential operations, while diamond nodes represent passing datasets. Modular steps, such as time binning or other processing steps, are contained within dashed boxes, indicating they are optional components that users can choose based on the satellite mission. Each step of the processing pipeline is described in more detail in the section later on.

The processing pipeline consists of three main stages: data extraction, processing, and output standardization.
For each satellite mission, a processing script, called a *recipe*, is written that defines the specific extraction information and processing steps for that mission.
Such a recipe is usually 100-200 lines long and consists of common building blocks that can be reused across different missions.
How processed data is saved to files can be easily adapted, enabling the smooth integration of data processed by EL-PASO into existing analysis codes that expect the data in a specific format.  

### Figure 1: EL-PASO Processing Pipeline

<img src="figures/drawio.png" alt="Processing Pipeline Flowchart" width="800"/>

*Figure 1: Schematic representation of the EL-PASO processing pipeline. Diamonds represent data, while rectangles represent processing steps. Optional steps and data are indicated with dashed boxes.*

## 2.1 Core Data Structure

The central data structure within EL-PASO is the custom *Variable* class.
This class serves as the canonical data container, holding a NumPy array of particle data along with comprehensive metadata (units, source files, description, processing notes). Similar to a NetCDF or CDF variable, the variable class crucially includes built-in unit handling using the Astropy physical units library (Astropy Collaboration et al., 2013, 2018, 2022). This integration ensures correct dimensional analysis and facilitates unit conversions automatically throughout the processing pipeline.

## 2.2 Data Extraction

The pipeline begins with data acquisition: EL-PASO can download files from common online repositories for a given time frame, or use files provided directly by the user.
The code snippet displayed in Figure 2 shows an example of downloading data from a public online repository.
As shown in the initial block of the snippet, the user defines the start and end times of the desired period, along with information about the online repository: URL, file names, and file cadence.
EL-PASO's download function will attempt to download files with the specified name from the provided URL.
If the file cadence is set to ''daily'', one file will be downloaded per day within the time period as defined by the start and end times.
The cadence can also be set to ''monthly'' or ''single_file''.

Online repositories often organize files by year, month, and day.
The user can encode this information in URLs and file names using custom placeholders such as ''YYYY'', ''MM'', or ''YYYYMMDD''.
The download function will replace these placeholders with the appropriate year or month numbers (e.g., ''YYYY'' will be resolved as ''2013'').
Similarly, the information about the version can be encoded in the file name using a regular expression.
The download function can detect semantic versioning this way and will only download the most up-to-date file (e.g., ''.\{6\}'' resolves to ''v8.4.0'').
Before downloading a file with a specific name, the download function checks whether a file with the same name already exists at the provided save path, unless the user sets the flag ''skip_existing'' to False.

Finally, the user also chooses a method to fetch the data.
By the time of this publication, the implemented methods use the ''request'' or ''wget'' module, or download the data from the ESA Space Weather Service Network using its API. 
All data files are stored under the provided save path and are ready for further processing.

### Figure 2: Code snippet to download data from an online repository.

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

import el_paso as ep

ep.setup_logging()

start_time = datetime(2013, 3, 17, 5, tzinfo=timezone.utc) # Inclusive start time of the processing window (UTC)
end_time = datetime(2013, 3, 17, 23, 59, tzinfo=timezone.utc)   # Inclusive end time of the processing window (UTC)
raw_data_path = Path("raw_data")                       # Raw downloaded files will be stored under this path
processed_data_path = Path("processed_data")           # Processed files will be stored under this path

# define the file organization in the online repository
file_name_stem = "rbspa_rel04_ect-hope-pa-l3_YYYYMMDD_.{6}.cdf"
download_url = "https://spdf.gsfc.nasa.gov/pub/data/rbsp/rbspa/l3/ect/hope/pitchangle/rel04/YYYY/"
file_cadence = "daily"

# download files from the online repository (file got already downloaded while starting up mybinder)
ep.download(
    start_time=start_time,
    end_time=end_time,
    save_path=raw_data_path,
    download_url=download_url,
    file_name_stem=file_name_stem,
    file_cadence=file_cadence,
    skip_existing=True,
    method="request",               # uses the 'request' library for downloading
)

## 2.3 Data Processing

Following extraction, the extracted data is manipulated, and derived quantities are calculated. In this step, the user implements mission-specific processing steps, although the most common routines are already available in EL-PASO, such as time binning or phase-space density calculation. The optional processing routines also include the calculation of adiabatic invariants ($\mu$, $J$, $L*$) (Schulz & Lanzerotti, 1974). This step uses global empirical magnetic field models, supplemented by local magnetic field measurements when available.
EL-PASO integrates the International Radiation Belt Environment Modeling (IRBEM) library to perform the heavy lifting of calculating adiabatic invariants and other calculations using magnetic field models. While the IRBEM library is written in Fortran, a custom Python wrapper is used to call its functions seamlessly from within EL-PASO. The calculation is parallelized using Python's multiprocessing module to speed up processing time.

A code snippet showing what these processing steps can look like is provided in Figure 3.
For the full description and documentation of the processing functions, we refer to EL-PASO's online documentation.

### Figure 3: Code snippet to extract variables from the downloaded files.

In [ ]:
# Extract variables from downloaded files
import astropy.units as u

# define how data is organized in the files
extraction_infos = [
    ep.ExtractionInfo(result_key="Epoch", name_or_column="Epoch_Ele", unit=ep.units.cdf_epoch),
    ep.ExtractionInfo(result_key="Energy", name_or_column="HOPE_ENERGY_Ele", unit=u.eV),
    ep.ExtractionInfo(result_key="Pitch_angle", name_or_column="PITCH_ANGLE", unit=u.deg, is_time_dependent=False),
    ep.ExtractionInfo(result_key="FEDU", name_or_column="FEDU", unit=(u.cm**2 * u.s * u.sr * u.keV) ** (-1)),
    ep.ExtractionInfo(result_key="xGEO", name_or_column="Position_Ele", unit=u.km),
]

# extract data from the downloaded files and store in a dictionary of variables
variables = ep.extract_variables_from_files(
    start_time=start_time,
    end_time=end_time,
    file_cadence=file_cadence,
    data_path=raw_data_path,
    file_name_stem=file_name_stem,
    extraction_infos=extraction_infos,
)

Following extraction, a variety of optional processing steps can be applied to the variables, such as time binning or the calculation of derived physical products (see Figure 4). The full up-to-date list of available processing steps can be found in the online documentation.

One crucial capability for radiation belt modeling is the calculation of adiabatic invariants ($\mu$, $J$, $L*$) (Schulz & Lanzerotti, 1974). This step utilizes global empirical magnetic field models and measurements of the local magnetic field, if available. EL-PASO integrates the International Radiation Belt Environment Modeling (IRBEM) library to perform the heavy lifting of calculating these adiabatic invariants and other calculations utilizing magnetic field models. While the IRBEM library is written in Fortran, a custom Python wrapper is used to call its functions seamlessly from within EL-PASO. The calculation is parallelized using Python's multiprocessing module to speed up the processing time.

### Figure 4: Code snippet to process the extracted variables and calculate derived products.

In [ ]:
# ------------- TIME BINNING -------------
# define options for time binning
time_bin_methods = {
    "xGEO": ep.TimeBinMethod.NanMean,
    "Energy": ep.TimeBinMethod.NanMedian,
    "FEDU": ep.TimeBinMethod.NanMedian,
    "Pitch_angle": ep.TimeBinMethod.Repeat,
}
time_variable = variables["Epoch"]
cadence_for_binning = timedelta(minutes=5)

# time bin all variables to a common time index
binned_time_variable = ep.processing.bin_by_time(
    time_variable=time_variable,
    variables=variables,
    time_bin_method_dict=time_bin_methods,
    time_binning_cadence=cadence_for_binning,
    start_time=start_time,
    end_time=end_time,
)

# making the flux dimensions consistent with PRBEM standard (time, energy, pitch angle)
variables["FEDU"].transpose_data([0, 2, 1])

# fold the flux around 90 degrees assuming symmetry 
ep.processing.fold_pitch_angles_and_flux(variables["FEDU"], variables["Pitch_angle"])  

# ------ MAGNETIC FIELD CALCULATIONS -----

irbem_options = ep.processing.magnetic_field_utils.IrbemOptions(
    ep.processing.magnetic_field_utils.LstarQuantity.NONE # change to Lstar to enable Lstar calculation (time consuming!)
)

# define options for magnetic field calculations
vars_to_compute = [
("B_Calc", "T89"), ("MLT", "T89"), ("B_Eq", "T89"), ("R_Eq", "T89"),
("Alpha_Eq", "T89"), ("L_star", "T89"), ("L_m", "T89"), ("InvK", "T89"), ("InvMu", "T89"),
]

# invoke IRBEM routines to calculate variables
magnetic_field_variables = ep.processing.compute_magnetic_field_variables(
    time_var=binned_time_variable,
    xgeo_var=variables["xGEO"],
    variables_to_compute=vars_to_compute,
    irbem_options=irbem_options,
    num_cores=1,
    pa_local_var=variables["Pitch_angle"],
    energy_var=variables["Energy"],
    particle_species="electron",
)

# compute phase space density from flux measurements
psd_var = ep.processing.compute_phase_space_density(
    flux_var=variables["FEDU"],
    energy_var=variables["Energy"],
    particle_species="electron"
)

## 2.4 Output Standardization

Upon completion of the processing steps, the resulting variables are saved into a standardized output format. This standardization is governed by two key components: a *SavingStrategy* and a *DataStandard*.

A saving strategy defines how the data is organized inside different output files. It is possible to split variables into separate files, choose a specific file format, and specify whether daily, monthly, or yearly files are created. Currently supported output formats are MATLAB MAT, NetCDF4, CDF, and HDF5. The data standard defines the naming conventions for variables and attributes, the dimension order, and the physical units to be used in the output files.

The library already includes base classes for the saving strategy and the data standard, which can be extended in order to create custom strategies and standards. For the up-to-date list of implemented saving strategies and data standards, please refer to the online documentation. At the time of this paper, a data standard used internally at GFZ is available, as well as part of the PRBEM data standard.

The code snippet in Figure 5 showcases how the processed variables are saved using monthly NetCDF4 files. The MonthlyRBStrategy is initialized using the mission, satellite, and instrument names, along with a data standard. Before saving, the variables are converted to the specified units and saved under the variable names defined in this data standard. Furthermore, some consistency checks are performed such as dimension sizes between variables sharing the same dimensions. EL-PASO's save function requires a dictionary as input that maps processed variables to standardized names, along with a saving strategy and a time period for saving. The save function is fully modular, meaning the same variables can be saved using a different saving strategy or data standards by simply replacing these arguments. This makes it possible to share recipes, which fully define how the data is processed, while the output format can still be modified to the individual needs.

### Figure 5: Code snippet to save the processed data into monthly NetCDF fles using the PRBEM standard.

In [ ]:
# Save processed data to standardized format

# use the strategy to sort the variables into monthly NetCDF files following the GFZ data standard
saving_strategy = ep.saving_strategies.MonthlyRBStrategy(
    base_data_path=processed_data_path,
    satellite="rbspa",
    mission="RBSP",
    instrument="hope",
    mag_field="T89",
    data_standard=ep.data_standards.GFZStandard(),
)

# define the mapping between our variables and the output variable names inside the files
variables_to_save: dict[ep.typing.InternalName, ep.Variable] = {
    "Epoch": binned_time_variable,
    "FEDU": variables["FEDU"],
    "Energy_FEDU": variables["Energy"],
    "Alpha": variables["Pitch_angle"],
    "Alpha_Eq": magnetic_field_variables["Alpha_Eq_T89"],
    "MLT": magnetic_field_variables["MLT_T89"],
    "B_Eq": magnetic_field_variables["B_Eq_T89"],
    "Position": variables["xGEO"],
    "L_star": magnetic_field_variables["L_star_T89"],
    "R_Eq": magnetic_field_variables["R_Eq_T89"],
}

# save the variables to disk according to the saving strategy
ep.save(
    variables_dict=variables_to_save,
    saving_strategy=saving_strategy,
    start_time=start_time,
    end_time=end_time,
    time_var=binned_time_variable
)

## 2.5 Loading of Processed Data

EL-PASO's *dataset* module allows for easy reading of data, which is saved by EL-PASO. Figure 6 shows two ways of loading data using the *Dataset* class. Once the Dataset got initialized using a saving strategy, the data can be accessed by using the ''get_by_interal_name'' method, which uses the variable names as used internally by EL-PASO or by accessing the attributes directly. The attribute names match the names as by the data standard. All data is loaded lazily which ensures that large amount of data can be loaded efficiently.

## Figure 6: Code snippet to load the saved data using EL-PASO’s dataset module

In [ ]:
hope_ds = ep.dataset.DataSet(saving_strategy, start_time, end_time, verbose=True)

from matplotlib import pyplot as plt
import numpy as np

plt.figure(figsize=(8,6))
plt.subplot(211)

# load data using internal names (type hints are available)
plt.plot(hope_ds.datetime, hope_ds.get_by_internal_name("R_Eq"), "k", label="R_eq")
plt.ylabel("Radial Distance [R_E]")

plt.subplot(212)

# load data using names as defined in the data standard
plt.plot(hope_ds.datetime, np.rad2deg(hope_ds.alpha_eq_model[:,-1]), "r:", label="Equatorial pitch angle")
plt.ylabel("Equatorial Pitch angle [°]")
plt.tight_layout()

---
# 3. Usability, Reproducibility, and Maintenance

## 3.1 Getting started

Installing EL-PASO requires a single *pip install* command. It is available through the Python Package Index (PyPI; *pip install el-paso*) or can be installed from a local clone *pip install path_to_el_paso*. The compilation of the IRBEM library is part of the EL-PASO installation process. The installation can be verified by running the minimal example provided in the EL-PASO repository.

After the installation has been verified, it is recommended to explore the tutorials, which consist of Jupyter notebooks. The tutorials cover the basic usage of EL-PASO, as well as more advanced topics, such as running EL-PASO efficiently on HPC clusters.

## 3.2 Reproducibility

A core asset of EL-PASO is the seamless reproducibility of processed data, enabled by the inherent architecture of its ''recipes''. Each recipe functions as a self-contained, self-documenting script that executes data processing in a fully deterministic manner. The long-term vision for EL-PASO is to serve as a comprehensive, community-driven repository of recipes spanning all major satellite missions. Currently, the framework includes 14 operational recipes. These cover diverse missions across the inner magnetosphere, including Low Earth Orbit (LEO), Medium Earth Orbit (MEO), and Geosynchronous Earth Orbit (GEO). All processed data can be read via the *dataset* module, streamlining downstream analysis and utilization.

## 3.3 Metadata and FAIR Principles

One of the main goals of EL-PASO is to provide metadata for processed data, enabling the reproducibility of scientific results and facilitating data sharing. The metadata is stored in the output files, while its exact format depends on the chosen file format. Some formats, such as NetCDF, allow for metadata stored at the file level and metadata for each variable, while other formats do not include a native way to include metadata (MAT, ASCII).

Importantly, the metadata content is not defined by EL-PASO; it only offers a tool to help store the metadata as desired by the user. All variables share a common set of metadata attributes that represent the basic metadata: the EL-PASO version used, the source files used for processing, the processing steps applied, the physical units, and variable descriptions. The predefined processing steps of EL-PASO add a processing note to the variable's metadata, ensuring that other users of the data can understand how it has been processed. An example of the metadata stored for a processed unidirectional differential electron flux variable is shown in Table 1.
Users can adjust the metadata of each variable to their needs by directly changing the variable's metadata fields or by implementing a custom saving strategy that handles metadata.  

In addition to metadata for each variable, for some file formats, such as NetCDF, the user can also define metadata at the file level, which is stored in all files. The combination of file-level and variable-level metadata enables the data to comply with metadata standards such as the SPASE Data Model. The stored metadata follows the recommendations for FAIR data principles (Wilkinson et al., 2016), making the processed data Findable, Accessible, Interoperable, and Reusable.

### Table 1: Metadata for processed unidirectional differential electron flux

| Field            | Value                                                                                                                                                                                                                                                                                                       |
|------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| units            | 1 / (keV s sr cm2)                                                                                                                                                                                                                                                                                          |
| source           | rbspa\_ect-elec-L3\_20170908\_v1.0.0.cdf                                                                                                                                                                                                                                                                     |
| processing notes | This variable was processed using the EL-PASO release mode on 19-Dec-2025 <br> User name: Bernhard Haas <br> Email address: bhaas@gfz.de <br> EL-PASO version: 0.1, git commit: f34736f36b0550922029cbfc7ec1636e6392e96e <br> 1) Time binned with method NanMedian and cadence of 60.0 minutes <br> 2) Folded around 90 degrees local pitch angle |
| description      | Processed unidirectional differential electron flux                                                                                                                                                                                                                                                       |

Furthermore, EL-PASO can be run in *release mode*, which ensures that all your local changes to the EL-PASO code are committed to git, allowing a unique commit hash to be stored in the metadata. 

---
# 4. Verification and Example Applications

We performed several verification tests against results obtained from other sources to ensure the correctness of the EL-PASO library. We verified the processing pipeline of calculating the derived product $L^*$, which is associated with the third adiabatic invariant, by comparing against published values by the Van Allen Probes ECT team (Spence et al., 2013). Furthermore, we tested the whole processing pipeline, from download to saving, by reproducing previous results obtained with software developed at the GFZ Helmholtz Centre for Geosciences and the University of California, Los Angeles (UCLA).

<div id="4.1-Reproducing-L-values-obtained-by-the-ECT-team"/>

## 4.1 Reproducing $L*$ values obtained by the ECT team

For this specific verification, the goal is to reproduce the $L^*$ product, which is calculated by the ECT team using their own software and then made publicly available. We use EL-PASO to download the Helium-Oxygen-Proton-Electron (HOPE) data (Funsten et al., 2013) from the Van Allen Probes mission for March 17, 2013. We extract the necessary variables, calculate $L^*$ using the IRBEM library, and save the results into a standardized output format.
The resulting $L^*$ values are then compared against the values obtained from the ECT team for the T89 (Tsyganenko, 1989) and TS04 (Tsyganenko & Sitnov, 2005) magnetic field models, and displayed in Figure 7.

### Figure 7: Comparison of L* values

<img src="figures/mageph_merged.png" alt="L* Comparison" width="800"/>

*Figure 7: Comparison of the $L*$ values obtained from EL-PASO and the ECT team for different pitch angles and magnetic field models.*

During this day, geomagnetic activity is enhanced as Kp reaches 7-, while Dst reaches a minimum of around -\SI{130}{\nano\tesla} (see Panels 7a and e). Panels 7b-d show the calculated $L^*$ values utilizing the T89 magnetic field model for 20°, 50°, and 70° local pitch angle, respectively. Panels 7f-h display the $L^*$ values using TS04 for the same pitch angles. The obtained root-mean-square-errors (RMSE) are relatively small ($<$ 0.1) for all pitch angles and magnetic field models, and are comparable to errors obtained for other studies, when different libraries for $L^*$-calculation are compared  (Min et al., 2013). Nevertheless, some minor discrepancies are worth mentioning: the ECT team uses a smoothed version of the Kp index, so the index does not suddenly change every three hours but changes gradually. This is visible around March 6:00 UTC, where the calculated $L^*$ using T89 changes gradually in the ECT data, while EL-PASO shows a sudden jump due to the change in Kp from 2+ to 7- at 6:00 UTC.

To extend this assessment, we further investigate the performance of the $L^*$ calculation utilizing the TS04 magnetic field model by comparing results for a 6-month period in early 2017 (see Figure 8). This reveals that the $L^*$ calculated by EL-PASO has a small mean error (ME) compared to the ECT result (see Panel 8d), while RMSE is still below 0.1. The bias could be caused by different implementations of the $L^*$ calculation itself or calculation of the $W$ input parameters, needed for the TS04 model (Tsyganenko & Sitnov, 2005). As the ECT team's processing software is not publicly available, we cannot investigate these discrepancies further. However, the discrepancies are not large enough to be concerning, as the uncertainty of $L^*$ due to the uncertainties in the magnetic field model is much larger (Thompson et al., 2021). We also observe some visual gaps in the $L^*$ calculated by EL-PASO, where the framework fails to calculate $L^*$ and gives NaN-values. We identified that these times correspond to missing values in the OMNI database, which is used in this work to retrieve solar wind parameters at the bow shock. If these NaN values cause issues during further analysis, one could also use measurements at L1 directly and propagate them manually to the bow shock.

### Figure 8: Long-term comparison of the L∗  values

<img src="figures/mag_eph_test_TS04_long_term.png" alt="Long-term L* Comparison" width="800"/>

*Figure 8: Long-term comparison of the $L^*$ values obtained from EL-PASO and the ECT team utilizing the TS04 magnetic field model. a) Kp and Dst time series. {b) $L^*$ values calculated by the ECT team for 70° local pitch angle. c) $L^*$ values calculated by EL-PASO for 70° local pitch angle. d) $L^*$ calculated by EL-PASO minus $L^*$ calculated by the ECT team.*

We consider this comparison a successful verification of EL-PASO's ability to calculate $L^*$. The small discrepancies we found underscore the need for open-source tools like EL-PASO, where all processing steps are transparent and verifiable. This enhances reproducibility in radiation belt studies.

## 4.2 Reproducing previous results obtained with other software developed at GFZ

Over the last 10 years, MATLAB code has been developed at GFZ and UCLA to process particle data from different missions. This code was used for a number of publications (e.g., Shprits et al., 2015, 2023; Drozdov et al., 2017, 2023; Himmelsbach et al., 2025). As this code has matured over the years, we consider it sufficiently trustworthy to use it as a reference for verifying EL-PASO.

For this verification test, we specifically aim to reproduce the processed data from the HOPE instrument for the March 2013 geomagnetic storm.
This data was previously processed using the MATLAB code and published in Haas et al. (2023). We use EL-PASO to download the HOPE data for the same time frame, extract the necessary variables, apply the same processing steps as in the MATLAB code, and save the results. The resulting processed data is then compared against the data obtained from the MATLAB code and displayed in Figure 9.

Panels 9b-d show the comparison of electron fluxes for different energies along with their RMSE values in units of degree of magnitude (dex). The fluxes agree well for all energies, as indicated by the low RMSEs, while the only differences can be observed when the satellite flies very close to Earth, where the GFZ-UCLA version gives NaN values, while EL-PASO provides reasonable fluxes. This is because the GFZ-UCLA version is more conservative with missing values and requires at least 50% of the points within each time bin to be valid during time binning. While EL-PASO has the same functionality, it has not been applied to this dataset, as the measured values near Earth seem reasonable.

The verification of the calculated adiabatic invariants and phase space densities, utilizing the TS04 magnetic field model, is displayed in Panels 9f-h. For all three quantities, we see a one-to-one agreement, further verifying the performance of EL-PASO. 

### Figure 9: Comparison with GFZ-UCLA Matlab code

<img src="figures/old_GFZ_merged.png" alt="drawing" width="800"/>

*Figure 9: Comparison of the fluxes, adiabatic invariants, and phase space densities obtained from the Matlab code developed at GFZ and UCLA and EL-PASO for different energy channels.*

---
# 5. Summary

In this paper, we presented EL-PASO, an open-source Python library for processing and standardizing in-situ particle measurements taken in space.
EL-PASO is self-contained, fully annotated using type hints, and complies with the PEP-8 standard. EL-PASO provides a robust, extensible library for processing particle data from various missions. Its modular design allows the easy addition of new missions, data formats, and processing steps, making it useful across a wide range of space physics applications. It is already in use at GFZ to prepare data for scientific publications and serves as the backbone of the data-processing pipeline for GFZ's data-assimilative radiation belt forecast.

The development of EL-PASO is an ongoing effort, and we welcome community contributions to enhance its capabilities and expand its reach. It will be extended in the future to include more satellite missions and incorporate new data standards once they become available.

---
# Open Research Section

All RBSP-ECT data are publicly available at the website: https://rbsp-ect.newmexicoconsortium.org/data_pub/rbspb/hope/level3/pitchangle/.

Dst and Kp values are from the NASA OMNIWeb data explorer, accessible at: https://omniweb.gsfc.nasa.gov/form/dx1.html

# Acknowledgments

Processing and analysis of the HOPE and ECT data were supported by the Energetic Particle, Composition, and Thermal Plasma (RBSP-ECT) investigation, funded under NASA's Prime contract no. NAS5-01072. All RBSP-ECT data are publicly available at the website https://rbsp-ect.newmexicoconsortium.org/data_pub/. We acknowledge the use of the IRBEM library, the latest version of which can be found at https://doi.org/10.5281/zenodo.6867552, and the use of AI tools for refining the manuscript language. This work has been partially funded by the German Research Foundation (NFDI4Earth, DFG project no. 460036893, https://www.nfdi4earth.de/) and DFG project no. 318763901 - SFB1294. This work was supported by the European Research Council (ERC) under the European Union's Horizon 2020 research and innovative programme (grant agreement number: 101124679-WIRE).

# Conflict of Interest Disclosure

The authors declare there are no conflicts of interest for this manuscript.

1. Angelopoulos, V., Cruce, P., Drozdov, A., Grimes, E. W., Hatzigeorgiu, N., King, D. A., … Schroeder, P. (2019, January). The Space Physics Environment Data Analysis System (SPEDAS). *Space Science Reviews, 215*(1), 9. Retrieved December 19, 2025, from [https://doi.org/10.1007/s11214-018-0576-4](https://doi.org/10.1007/s11214-018-0576-4). DOI: [10.1007/s11214-018-0576-4](https://doi.org/10.1007/s11214-018-0576-4)
2. Astropy Collaboration, Price-Whelan, A. M., Lim, P. L., Earl, N., Starkman, N., Bradley, L., … Astropy Contributors. (2022). The Astropy Project: Sustaining and Growing a Community-oriented Open-source Project and the Core Python Package for Astronomy. *The Astrophysical Journal, 935*, 167. DOI: [10.3847/1538-4357/ac7c74](https://doi.org/10.3847/1538-4357/ac7c74)
3. Astropy Collaboration, Price-Whelan, A. M., Sipőcz, B. M., Günther, H. M., Lim, P. L., Crawford, S. M., … Astropy Contributors. (2018). The Astropy Project: Building an Open-science Community and an Ecosystem of Astronomical Software in Python. *The Astronomical Journal, 156*, 123. DOI: [10.3847/1538-3881/aabc4f](https://doi.org/10.3847/1538-3881/aabc4f)
4. Astropy Collaboration, Robitaille, T. P., Tollerud, E. J., Greenfield, P., Droettboom, M., Bray, E., … Streicher, O. (2013). Astropy: A Community Python Package for Astronomy. *Astronomy & Astrophysics, 558*, A33. DOI: [10.1051/0004-6361/201322068](https://doi.org/10.1051/0004-6361/201322068)
5. Christie, T., et al. (2014). MkDocs: Project documentation with Markdown. Retrieved from [https://www.mkdocs.org/](https://www.mkdocs.org/)
6. Das2: Interface Control Document. (n.d.). Retrieved from [https://zenodo.org/records/3588535](https://zenodo.org/records/3588535). DOI: [10.5281/zenodo.3588535](https://doi.org/10.5281/zenodo.3588535)
7. Drozdov, A. Y., Kondrashov, D., Strounine, K., & Shprits, Y. Y. (2023). Reconstruction of electron radiation belts using data assimilation and machine learning. *Frontiers in Astronomy and Space Sciences, 10*. Retrieved June 21, 2023, from [https://www.frontiersin.org/articles/10.3389/fspas.2023.1072795](https://www.frontiersin.org/articles/10.3389/fspas.2023.1072795)
8. Drozdov, A. Y., Shprits, Y. Y., Usanova, M. E., Aseev, N. A., Kellerman, A. C., & Zhu, H. (2017). EMIC wave parameterization in the long-term VERB code simulation. *Journal of Geophysical Research: Space Physics, 122*(8), 8488–8501. Retrieved February 20, 2023, from [https://onlinelibrary.wiley.com/doi/abs/10.1002/2017JA024389](https://onlinelibrary.wiley.com/doi/abs/10.1002/2017JA024389) (Eprint: [https://onlinelibrary.wiley.com/doi/pdf/10.1002/2017JA024389](https://onlinelibrary.wiley.com/doi/pdf/10.1002/2017JA024389)). DOI: [10.1002/2017JA024389](https://doi.org/10.1002/2017JA024389)
9. Funsten, H. O., Skoug, R. M., Guthrie, A. A., MacDonald, E. A., Baldonado, J. R., Harper, R. W., … Chen, J. (2013, November). Helium, Oxygen, Proton, and Electron (HOPE) Mass Spectrometer for the Radiation Belt Storm Probes Mission. *Space Science Reviews, 179*(1), 423–484. Retrieved January 5, 2026, from [https://doi.org/10.1007/s11214-013-9968-7](https://doi.org/10.1007/s11214-013-9968-7). DOI: [10.1007/s11214-013-9968-7](https://doi.org/10.1007/s11214-013-9968-7)
10. Grimes, E. W., Harter, B., Hatzigeorgiu, N., Drozdov, A., Lewis, J. W., Angelopoulos, V., … Le Contel, O. (2022, October). The Space Physics Environment Data Analysis System in Python. *Frontiers in Astronomy and Space Sciences, 9*. Retrieved December 19, 2025, from [https://www.frontiersin.org/journals/astronomy-and-space-sciences/articles/10.3389/fspas.2022.1020815/full](https://www.frontiersin.org/journals/astronomy-and-space-sciences/articles/10.3389/fspas.2022.1020815/full). DOI: [10.3389/fspas.2022.1020815](https://doi.org/10.3389/fspas.2022.1020815)
11. Haas, B., Shprits, Y. Y., Allison, H. J., Wutzig, M., & Wang, D. (2023, January). A missing dusk-side loss process in the terrestrial electron ring. *Scientific Reports, 13*(1), 970. Retrieved April 20, 2023, from [https://www.nature.com/articles/s41598-023-28093-2](https://www.nature.com/articles/s41598-023-28093-2). DOI: [10.1038/s41598-023-28093-2](https://doi.org/10.1038/s41598-023-28093-2)
12. Harris, C. R., Millman, K. J., van der Walt, S. J., Gommers, R., Virtanen, P., Cournapeau, D., … Oliphant, T. E. (2020, September). Array programming with NumPy. *Nature, 585*(7825), 357–362. Retrieved from [https://doi.org/10.1038/s41586-020-2649-2](https://doi.org/10.1038/s41586-020-2649-2). DOI: [10.1038/s41586-020-2649-2](https://doi.org/10.1038/s41586-020-2649-2)
13. Himmelsbach, J., Shprits, Y. Y., Allison, H., Haas, B., Wutzig, M., Szabo-Roberts, M., … Drozdov, A. Y. (2025). Using VERB-4D to Model Ring Current Ions and Their Sensitivity to Plasmasphere Density. *Journal of Geophysical Research: Space Physics, 130*(3), e2024JA033128. Retrieved March 27, 2025, from [https://onlinelibrary.wiley.com/doi/abs/10.1029/2024JA033128](https://onlinelibrary.wiley.com/doi/abs/10.1029/2024JA033128) (Eprint: [https://onlinelibrary.wiley.com/doi/pdf/10.1029/2024JA033128](https://onlinelibrary.wiley.com/doi/pdf/10.1029/2024JA033128)). DOI: [10.1029/2024JA033128](https://doi.org/10.1029/2024JA033128)
14. Krekel, H., Oliveira, B., Pfannschmidt, R., Bruynooghe, F., Laugher, B., & Bruhin, F. (2004). *pytest*. Retrieved from [https://github.com/pytest-dev/pytest](https://github.com/pytest-dev/pytest) (Contributors include Holger Krekel, Bruno Oliveira, Ronny Pfannschmidt, Floris Bruynooghe, Brianna Laugher, Florian Bruhin, and others.)
15. Marsh, C., & Astral Shaper, Inc. (2022). *Ruff: An extremely fast Python linter and code formatter*. Retrieved from [https://github.com/astral-sh/ruff](https://github.com/astral-sh/ruff)
16. Mauk, B. H., Fox, N. J., Kanekal, S. G., Kessel, R. L., Sibeck, D. G., & Ukhorskiy, A. (2013, November). Science Objectives and Rationale for the Radiation Belt Storm Probes Mission. *Space Science Reviews, 179*(1), 3–27. Retrieved June 28, 2023, from [https://doi.org/10.1007/s11214-012-9908-y](https://doi.org/10.1007/s11214-012-9908-y). DOI: [10.1007/s11214-012-9908-y](https://doi.org/10.1007/s11214-012-9908-y)
17. Microsoft Corporation. (2019). *Pyright: Static type checker for Python*. Retrieved from [https://github.com/microsoft/pyright](https://github.com/microsoft/pyright)
18. Morley, S. K., Koller, J., Welling, D. T., Larsen, B. A., Henderson, M. G., & Niehof, J. T. (2011). *Spacepy – A Python-based library of tools for the space sciences*. In *Proceedings of the 9th Python in Science Conference (SciPy 2010)*, Austin, TX.
19. Niehof, J. T., Morley, S. K., Welling, D. T., & Larsen, B. A. (2022). The spacepy space science package at 12 years. *Frontiers in Astronomy and Space Sciences, 9*. DOI: [10.3389/fspas.2022.1023612](https://doi.org/10.3389/fspas.2022.1023612)
20. pandas development team, T. (2020, February). *pandas-dev/pandas: Pandas*. Zenodo. Retrieved from [https://doi.org/10.5281/zenodo.3509134](https://doi.org/10.5281/zenodo.3509134). DOI: [10.5281/zenodo.3509134](https://doi.org/10.5281/zenodo.3509134)
21. Schulz, M., & Lanzerotti, L. J. (1974). Pitch-Angle Diffusion. In *Particle Diffusion in the Radiation Belts* (pp. 46–80). Berlin, Heidelberg: Springer Berlin Heidelberg. Retrieved from [https://doi.org/10.1007/978-3-642-65675-0_3](https://doi.org/10.1007/978-3-642-65675-0_3). DOI: [10.1007/978-3-642-65675-0_3](https://doi.org/10.1007/978-3-642-65675-0_3)
22. Shprits, Y. Y., Kellerman, A. C., Drozdov, A. Y., Spence, H. E., Reeves, G. D., & Baker, D. N. (2015, November). Combined convective and diffusive simulations: VERB-4D comparison with 17 March 2013 Van Allen Probes observations. *Geophysical Research Letters, 42*(22), 9600–9608. DOI: [10.1002/2015GL065230](https://doi.org/10.1002/2015GL065230)
23. Shprits, Y. Y., Michaelis, I., Wang, D., Allison, H., Vasile, R., Runov, A., … Smirnov, A. (2023). MLT Dependence of Relativistic Electron Scattering Into the Drift Loss Cone: Measurements From ELFIN-L on Board Lomonosov Spacecraft. *Geophysical Research Letters, 50*(12), e2023GL103342. Retrieved January 21, 2025, from [https://onlinelibrary.wiley.com/doi/abs/10.1029/2023GL103342](https://onlinelibrary.wiley.com/doi/abs/10.1029/2023GL103342) (Eprint: [https://onlinelibrary.wiley.com/doi/pdf/10.1029/2023GL103342](https://onlinelibrary.wiley.com/doi/pdf/10.1029/2023GL103342)). DOI: [10.1029/2023GL103342](https://doi.org/10.1029/2023GL103342)
24. Spence, H. E., Reeves, G. D., Baker, D. N., Blake, J. B., Bolton, M., Bourdarie, S., … Thorne, R. M. (2013, November). Science Goals and Overview of the Radiation Belt Storm Probes (RBSP) Energetic Particle, Composition, and Thermal Plasma (ECT) Suite on NASA's Van Allen Probes Mission. *Space Science Reviews, 179*(1), 311–336. Retrieved January 5, 2026, from [https://doi.org/10.1007/s11214-013-0007-5](https://doi.org/10.1007/s11214-013-0007-5). DOI: [10.1007/s11214-013-0007-5](https://doi.org/10.1007/s11214-013-0007-5)
25. Tsyganenko, N. A. (1989). A magnetospheric magnetic field model with a warped tail current sheet. *Planetary and Space Science, 37*(1), 5–20. DOI: [10.1016/0032-0633(89)90066-4](https://doi.org/10.1016/0032-0633(89)90066-4)
26. Tsyganenko, N. A., & Sitnov, M. I. (2005). Modeling the dynamics of the inner magnetosphere during strong geomagnetic storms. *Journal of Geophysical Research: Space Physics, 110*(A3). Retrieved April 26, 2022, from [https://onlinelibrary.wiley.com/doi/abs/10.1029/2004JA010798](https://onlinelibrary.wiley.com/doi/abs/10.1029/2004JA010798). DOI: [10.1029/2004JA010798](https://doi.org/10.1029/2004JA010798)
27. Virtanen, P., Gommers, R., Oliphant, T. E., Haberland, M., Reddy, T., Cournapeau, D., … SciPy 1.0 Contributors. (2020). SciPy 1.0: Fundamental algorithms for scientific computing in Python. *Nature Methods, 17*, 261–272. DOI: [10.1038/s41592-019-0686-2](https://doi.org/10.1038/s41592-019-0686-2)
28. Weigel, R. S., Vandegri, J., Faden, J., King, T., Roberts, D. A., Harris, B., … Martinez, B. (2021). HAPI: An API Standard for Accessing Heliospheric Time Series Data. *Journal of Geophysical Research: Space Physics, 126*(12), e2021JA029534. Retrieved January 15, 2026, from [https://onlinelibrary.wiley.com/doi/abs/10.1029/2021JA029534](https://onlinelibrary.wiley.com/doi/abs/10.1029/2021JA029534) (Eprint: [https://agupubs.onlinelibrary.wiley.com/doi/pdf/10.1029/2021JA029534](https://agupubs.onlinelibrary.wiley.com/doi/pdf/10.1029/2021JA029534)). DOI: [10.1029/2021JA029534](https://doi.org/10.1029/2021JA029534)
29. Wilkinson, M. D., Dumontier, M., Aalbersberg, I. J., Appleton, G., Axton, M., Baak, A., … Mons, B. (2016, March). The FAIR Guiding Principles for scientific data management and stewardship. *Scientific Data, 3*(1), 160018. Retrieved January 5, 2026, from [https://www.nature.com/articles/sdata201618](https://www.nature.com/articles/sdata201618). DOI: [10.1038/sdata.2016.18](https://doi.org/10.1038/sdata.2016.18)